# Silent Push API — Comprehensive Reference Notebook

![Python 3.9+](https://img.shields.io/badge/python-3.9%2B-blue?logo=python&logoColor=white) ![Jupyter](https://img.shields.io/badge/Jupyter-notebook-orange?logo=jupyter&logoColor=white)
This notebook covers all standard Silent Push APIs used for integrations.  
All endpoints are current as of the latest KB and standard integration list.  
Run **Section 0** first to configure authentication and shared helpers before running any other section.

**Base URLs:**
- Explore / PADNS / Feed APIs: `https://app.silentpush.com`
- ThreatCheck API: `https://api.threatcheck.silentpush.com`

**Authentication:**
- Explore APIs: `x-api-key` request header
- ThreatCheck API: `u` query parameter (Access Key)

---

| # | Section | APIs Covered |
|:---:|:---|---|
| 0 | Setup & Authentication | Imports, credentials, helper |
| 1 | ThreatCheck API | IOFA, Traffic Origin |
| 2 | Data Enrichment | Domain, IPv4, IPv6 (single + bulk) |
| 3 | Domain Intelligence | Domain Search, WHOIS, Certificates, Risk Score |
| 4 | Passive DNS (PADNS) | Forward, Reverse, Multi-condition, Density, IP Diversity, ASNs |
| 5 | Reputation & Risk History | IPv4, Nameserver, Subnet |
| 6 | SPQL Query Language | SPQL Search |
| 7 | Live Scanning | Live Scan v2, Live Screenshot v2 |
| 8 | Data Exports | IOFA, Organization, Bulk Data, IP Context |
| 9 | Customer Feed Management | Create Feed, Add Indicators, Add Tags |

---
## Section 0 — Setup & Authentication

> [!IMPORTANT]
> Run **Section 0** before anything else. Every subsequent section depends on the credentials, base URLs, and helper functions (`sp_request`, `to_dataframe`, `save_export`) defined here.

In [ ]:
import os
import time
import getpass
import json
import requests
import pandas as pd
from datetime import datetime
from IPython.display import display, Image

# ── Credentials ───────────────────────────────────────────────────────────────
# Reads from environment variables if set; otherwise prompts securely.
# To avoid prompts, set in your shell before launching Jupyter:
#   export SP_API_KEY="your-key"
#   export SP_ACCESS_KEY="your-access-key"
API_KEY    = os.environ.get("SP_API_KEY")    or getpass.getpass("Enter Silent Push API Key: ")
ACCESS_KEY = os.environ.get("SP_ACCESS_KEY") or getpass.getpass("Enter ThreatCheck Access Key: ")

# ── Base URLs ─────────────────────────────────────────────────────────────────
BASE_URL         = "https://app.silentpush.com"
THREATCHECK_BASE = "https://api.threatcheck.silentpush.com"

# ── Shared headers for Explore / PADNS / Feed / SPQL APIs ─────────────────────
HEADERS = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}

# ── Helper: make a request with error handling and auto-retry on rate limits ───
def sp_request(method, url, retries=3, **kwargs):
    """
    Make a Silent Push API request. Handles:
      - 429 rate limits (exponential backoff, up to `retries` attempts)
      - 401 auth errors (prints a clear message)
      - Non-JSON responses (prints raw text instead of crashing)
    Returns parsed JSON dict on success, None on failure.
    """
    kwargs.setdefault("timeout", 30)
    for attempt in range(retries):
        resp = requests.request(method, url, headers=HEADERS, **kwargs)
        if resp.status_code == 429:
            wait = 2 ** attempt
            print(f"Rate limited (attempt {attempt + 1}/{retries}). Retrying in {wait}s...")
            time.sleep(wait)
            continue
        if resp.status_code == 401:
            print("Authentication failed — check your API_KEY.")
            return None
        try:
            data = resp.json()
        except Exception:
            print(f"Non-JSON response (HTTP {resp.status_code}):\n{resp.text[:500]}")
            return None
        print(f"Status: {resp.status_code}")
        print(json.dumps(data, indent=2))
        return data
    print("Max retries exceeded.")
    return None

# ── Helper: extract a response array and display as a DataFrame ───────────────
def to_dataframe(data, path=None):
    """
    Extract an array from a response dict and display it as a pandas DataFrame.
    `path` is an optional dot-separated key path into the response,
    e.g. path="response.records" for {"response": {"records": [...]}}.
    If omitted, tries the top-level value if it's a list.
    """
    if data is None:
        return None
    obj = data
    if path:
        for key in path.split("."):
            if isinstance(obj, dict):
                obj = obj.get(key, {})
    # Auto-detect: look for any top-level list value
    if not isinstance(obj, list):
        for v in (data.get("response", {}) if isinstance(data, dict) else {}).values():
            if isinstance(v, list):
                obj = v
                break
    if isinstance(obj, list) and obj:
        df = pd.json_normalize(obj)
        display(df)
        return df
    print("No tabular data found in response to display as DataFrame.")
    return None

# ── Helper: save export content with an auto-generated timestamped filename ────
def save_export(content, base_name):
    """
    Save binary export content to disk with a timestamp in the filename
    to prevent overwriting previous downloads.
    Returns the filename saved.
    """
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"{base_name}_{ts}.csv"
    with open(fname, "wb") as f:
        f.write(content)
    print(f"Saved: {fname} ({len(content):,} bytes)")
    return fname

print("Setup complete.")

In [ ]:
# ── Credential Validation ─────────────────────────────────────────────────────
# Runs a lightweight enrichment call to confirm your API key is working.
# If this fails, check your API_KEY before running any other section.
print("Validating API key...")
_val = sp_request("GET", f"{BASE_URL}/api/v1/merge-api/explore/enrich/domain/example.com")
if _val is not None:
    print("\nAPI key is valid. You're ready to run any section.")
else:
    print("\nAPI key validation failed. Re-run the setup cell with a valid key.")

---
## Section 1 — ThreatCheck API

Quickly determine if an IP address or hostname appears on a Silent Push threat feed.  
Uses a separate base URL (`api.threatcheck.silentpush.com`) and authenticates via the `u` query parameter (Access Key).  
Docs: https://help.silentpush.com/docs/threat-check-api-endpoints

> [!NOTE]
> ThreatCheck uses a **separate base URL** (`api.threatcheck.silentpush.com`) and authenticates via the `u` query parameter using your **Access Key** — not your API key. If you only have an API key, skip this section.


### 1.1 IOFA Threat Check

Determine if an IP or hostname is listed on a Silent Push **Indicator of Future Attack (IOFA)** feed.

```
GET https://api.threatcheck.silentpush.com/v1/?t={indicator_type}&d=iofa&u={access_key}&q={indicator}
```

| Parameter | Description | Example values |
|---|---|---|
| `t` | Indicator type | `ip`, `domain` |
| `d` | Data source | `iofa` |
| `u` | Access Key | your ACCESS_KEY |
| `q` | Indicator to check | `8.8.8.8`, `example.com` |

In [ ]:
# ── 1.1 IOFA Threat Check ─────────────────────────────────────────────────────
indicator_type = "ip"          # 'ip' or 'domain'
indicator      = "8.8.8.8"    # IP address or hostname to check

url = f"{THREATCHECK_BASE}/v1/"
params = {
    "t": indicator_type,
    "d": "iofa",
    "u": ACCESS_KEY,
    "q": indicator
}

# ThreatCheck uses query params for auth, not the shared HEADERS
resp = requests.get(url, params=params)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

### 1.2 Traffic Origin Threat Check

Determine if an IP returns **Traffic Origin** data (identifies hosting infrastructure used for malicious traffic).

```
GET https://api.threatcheck.silentpush.com/v1/?t={indicator_type}&d=trafficorigin&u={access_key}&q={indicator}
```

In [ ]:
# ── 1.2 Traffic Origin Threat Check ──────────────────────────────────────────
indicator_type = "ip"          # 'ip' or 'domain'
indicator      = "8.8.8.8"    # IP address or hostname to check

url = f"{THREATCHECK_BASE}/v1/"
params = {
    "t": indicator_type,
    "d": "trafficorigin",
    "u": ACCESS_KEY,
    "q": indicator
}

resp = requests.get(url, params=params)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

---
## Section 2 — Data Enrichment

Retrieve comprehensive enrichment data for domains, IPv4 addresses, and IPv6 addresses.  
Single-indicator endpoints return detailed risk scores, ASN info, reputation, geolocation, and more.  
Bulk endpoints accept up to 100 indicators per request.

### 2.1 Enrich a Domain or Hostname

Retrieve enrichment data for a single domain or hostname.  
Docs: https://help.silentpush.com/docs/domain-enrichment

```
GET https://app.silentpush.com/api/v1/merge-api/explore/enrich/domain/{domain}
```

In [ ]:
# ── 2.1 Enrich a Domain ───────────────────────────────────────────────────────
domain = "example.com"

url = f"{BASE_URL}/api/v1/merge-api/explore/enrich/domain/{domain}"
sp_request("GET", url)

### 2.2 Enrich an IPv4 Address

Retrieve enrichment data for a single IPv4 address.  
Docs: https://help.silentpush.com/docs/ipv4-enrichment

```
GET https://app.silentpush.com/api/v1/merge-api/explore/enrich/ipv4/{ipv4}
```

In [ ]:
# ── 2.2 Enrich an IPv4 Address ────────────────────────────────────────────────
ipv4 = "8.8.8.8"

url = f"{BASE_URL}/api/v1/merge-api/explore/enrich/ipv4/{ipv4}"
sp_request("GET", url)

### 2.3 Enrich an IPv6 Address

Retrieve enrichment data for a single IPv6 address.  
Docs: https://help.silentpush.com/docs/ipv6-enrichment

```
GET https://app.silentpush.com/api/v1/merge-api/explore/enrich/ipv6/{ipv6}
```

In [ ]:
# ── 2.3 Enrich an IPv6 Address ────────────────────────────────────────────────
ipv6 = "2001:4860:4860::8888"   # Google's public IPv6 DNS

url = f"{BASE_URL}/api/v1/merge-api/explore/enrich/ipv6/{ipv6}"
sp_request("GET", url)

### 2.4 Bulk Enrich Domains

Get enrichment details for up to **100 domains** in a single request.  
Docs: https://help.silentpush.com/docs/bulk-enrich-indicatora

```
POST https://app.silentpush.com/api/v1/merge-api/explore/bulk/summary/domain
```

Request body: JSON array of domain strings.

In [ ]:
# ── 2.4 Bulk Enrich Domains ───────────────────────────────────────────────────
domains = [
    "example.com",
    "google.com",
    "github.com"
    # Add up to 100 domains
]

url = f"{BASE_URL}/api/v1/merge-api/explore/bulk/summary/domain"
data = sp_request("POST", url, json=domains)
to_dataframe(data)  # Displays results as a table for easy comparison

### 2.5 Bulk Enrich IPv4 Addresses

Get enrichment details for up to **100 IPv4 addresses** in a single request.  
Docs: https://help.silentpush.com/docs/bulk-ipv4-enrichment

```
POST https://app.silentpush.com/api/v1/merge-api/explore/bulk/summary/ipv4
```

In [ ]:
# ── 2.5 Bulk Enrich IPv4 Addresses ───────────────────────────────────────────
ipv4_addresses = [
    "8.8.8.8",
    "1.1.1.1",
    "9.9.9.9"
    # Add up to 100 IPv4 addresses
]

url = f"{BASE_URL}/api/v1/merge-api/explore/bulk/ip2asn/ipv4"
data = sp_request("POST", url, json=ipv4_addresses)
to_dataframe(data)  # Displays results as a table for easy comparison

### 2.6 Bulk Enrich IPv6 Addresses

Get enrichment details for up to **100 IPv6 addresses** in a single request.  
Docs: https://help.silentpush.com/docs/bulk-ipv6-enrichment

```
POST https://app.silentpush.com/api/v1/merge-api/explore/bulk/summary/ipv6
```

In [ ]:
# ── 2.6 Bulk Enrich IPv6 Addresses ───────────────────────────────────────────
ipv6_addresses = [
    "2001:4860:4860::8888",
    "2606:4700:4700::1111"
    # Add up to 100 IPv6 addresses
]

url = f"{BASE_URL}/api/v1/merge-api/explore/bulk/ip2asn/ipv6"
data = sp_request("POST", url, json=ipv6_addresses)
to_dataframe(data)  # Displays results as a table for easy comparison

---
## Section 3 — Domain Intelligence

Search for domains by various criteria, retrieve WHOIS records and SSL certificates,  
and get bulk risk scores across a list of domains.

### 3.1 Domain Search

Search for domains matching a combination of name patterns, network criteria, registrar, and WHOIS filters.  
Supports wildcard patterns (e.g., `*.example.com`, `evil*`).  
Docs: https://help.silentpush.com/docs/domain-search

```
GET https://app.silentpush.com/api/v1/merge-api/explore/domain/search
```

| Parameter | Description | Example |
|---|---|---|
| `domain` | Domain name / wildcard pattern | `evil*`, `*.example.com` |
| `asnum` | ASN filter | `15169` |
| `registrar` | Registrar name filter | `GoDaddy` |
| `whois_date` | WHOIS registration date filter | `2024-01-01` |
| `first_seen` | First seen date filter | `2024-01-01` |
| `last_seen` | Last seen date filter | `2024-12-31` |
| `limit` | Max results (default 100) | `100` |

In [ ]:
# ── 3.1 Domain Search ─────────────────────────────────────────────────────────
params = {
    "domain": "evil*",     # Wildcard pattern for domain name
    # "asnum": "15169",    # Uncomment to filter by ASN
    # "registrar": "GoDaddy",  # Uncomment to filter by registrar
    # "first_seen": "2024-01-01",  # Uncomment to filter by first seen date
    "limit": 10
}

url = f"{BASE_URL}/api/v1/merge-api/explore/domain/search"
data = sp_request("GET", url, params=params)
to_dataframe(data)  # Displays matching domains as a table

### 3.2 WHOIS Information

Retrieve previously collected WHOIS information for a domain, including registrant, registrar, and key dates.  
Docs: https://help.silentpush.com/docs/whois-information

```
GET https://app.silentpush.com/api/v1/merge-api/explore/domain/whois/{domain}
```

In [ ]:
# ── 3.2 WHOIS Information ─────────────────────────────────────────────────────
domain = "example.com"

url = f"{BASE_URL}/api/v1/merge-api/explore/domain/whois/{domain}"
sp_request("GET", url)

### 3.3 Domain Certificates

Retrieve SSL/TLS certificates associated with a domain.  
Docs: https://help.silentpush.com/docs/domain-certificates

```
GET https://app.silentpush.com/api/v1/merge-api/explore/domain/certificates/{domain}
```

In [ ]:
# ── 3.3 Domain Certificates ───────────────────────────────────────────────────
domain = "example.com"

url = f"{BASE_URL}/api/v1/merge-api/explore/domain/certificates/{domain}"
sp_request("GET", url)

### 3.4 Bulk Silent Push Risk Score for Domains

Get the Silent Push Risk Score for multiple domains in a single request (up to 100).  
Docs: https://help.silentpush.com/docs/bulk-silent-push-risk-score-for-a-list-of-domains

```
POST https://app.silentpush.com/api/v1/merge-api/explore/bulk/domain/riskscore
```

Request body: JSON array of domain strings.

In [ ]:
# ── 3.4 Bulk Silent Push Risk Score for Domains ───────────────────────────────
domains = [
    "example.com",
    "google.com",
    "suspiciousdomain.xyz"
    # Add up to 100 domains
]

url = f"{BASE_URL}/api/v1/merge-api/explore/bulk/domain/riskscore"
data = sp_request("POST", url, json=domains)
to_dataframe(data)  # Displays risk scores as a sortable table

---
## Section 4 — Passive DNS (PADNS)

Query Silent Push's Passive DNS database for historical DNS records.  
Supports forward lookups (domain → IPs), reverse lookups (IP → domains),  
multi-condition queries, and aggregated views like density and IP diversity.

### 4.1 Forward PADNS Lookup

Forward lookup of Passive DNS data — find what a domain resolves to for a given DNS record type.  
Docs: https://help.silentpush.com/docs/forward-padns-lookup

```
GET https://app.silentpush.com/api/v1/merge-api/explore/padns/lookup/query/{qtype}/{qname}
```

| `qtype` options | Description |
|---|---|
| `a` | A records (IPv4) |
| `aaaa` | AAAA records (IPv6) |
| `cname` | CNAME records |
| `mx` | MX records |
| `ns` | NS records |
| `txt` | TXT records |
| `soa` | SOA records |
| `any` | All record types |

In [ ]:
# ── 4.1 Forward PADNS Lookup ──────────────────────────────────────────────────
qtype = "a"              # Record type: a, aaaa, cname, mx, ns, txt, soa, any
qname = "example.com"   # Domain to look up
limit = 100

url = f"{BASE_URL}/api/v1/merge-api/explore/padns/lookup/query/{qtype}/{qname}"
data = sp_request("GET", url, params={"limit": limit})
to_dataframe(data)  # Displays DNS records as a table with first/last seen timestamps

### 4.2 Reverse PADNS Lookup

Reverse lookup — find all domains that have resolved to a given IP address or server name.  
Docs: https://help.silentpush.com/docs/reverse-padns-lookup

```
GET https://app.silentpush.com/api/v1/merge-api/explore/padns/lookup/answer/{qtype}/{qname}
```

For reverse lookups, `qname` is typically an IP address or nameserver hostname.

In [ ]:
# ── 4.2 Reverse PADNS Lookup ──────────────────────────────────────────────────
qtype = "a"              # Record type to reverse-look up
qname = "93.184.216.34" # IP address or nameserver to reverse look up
limit = 100

url = f"{BASE_URL}/api/v1/merge-api/explore/padns/lookup/answer/{qtype}/{qname}"
data = sp_request("GET", url, params={"limit": limit})
to_dataframe(data)  # Displays co-hosted domains as a table

### 4.3 Multi-condition PADNS Lookup

Search Passive DNS records where **both** the query name and the answer match specified values.  
Supports wildcard patterns in both `qname` and `qanswer`.  
Docs: https://help.silentpush.com/docs/multi-condition-padns-lookup

```
GET https://app.silentpush.com/api/v1/merge-api/explore/padns/lookup/both/{qtype}/{qname}/{qanswer}
```

Example: Find all A records where the domain matches `*.example.*` AND the IP is in `192.0.2.*`

In [ ]:
# ── 4.3 Multi-condition PADNS Lookup ─────────────────────────────────────────
qtype   = "a"                # Record type
qname   = "*.example.com"   # Domain pattern (wildcards supported)
qanswer = "93.184.216.*"     # Answer pattern (wildcards supported)

url = f"{BASE_URL}/api/v1/merge-api/explore/padns/lookup/both/{qtype}/{qname}/{qanswer}"
sp_request("GET", url)

### 4.4 Density Lookup

Get the **density** (count of unique domains) for a given query type and value.  
Useful for identifying shared infrastructure — e.g., how many domains share a nameserver.  
Docs: https://help.silentpush.com/docs/density-lookup

```
GET https://app.silentpush.com/api/v1/merge-api/explore/padns/lookup/density/{qtype}/{query}
```

In [ ]:
# ── 4.4 Density Lookup ────────────────────────────────────────────────────────
qtype = "ns"                    # Record type to measure density for
query = "ns1.example.com"       # Nameserver or IP to measure

url = f"{BASE_URL}/api/v1/merge-api/explore/padns/lookup/density/{qtype}/{query}"
sp_request("GET", url)

### 4.5 IP Diversity Lookup

Get the **IP diversity** (count of unique IP addresses seen over time) for a given domain or query.  
High IP diversity can indicate fast-flux infrastructure.

```
GET https://app.silentpush.com/api/v1/merge-api/explore/padns/lookup/ipdiversity/{qtype}/{query}
```

In [ ]:
# ── 4.5 IP Diversity Lookup ───────────────────────────────────────────────────
qtype = "a"              # Record type
query = "example.com"   # Domain to measure IP diversity for

url = f"{BASE_URL}/api/v1/merge-api/explore/padns/lookup/ipdiversity/{qtype}/{query}"
sp_request("GET", url)

### 4.6 Search IP Diversity Patterns

Search for IP Diversity patterns across multiple domains, with optional nameserver and domain name pattern filtering.  
Docs: https://help.silentpush.com/docs/search-ipdiversity-patterns

```
GET https://app.silentpush.com/api/v1/merge-api/explore/padns/search/ipdiversity
```

| Parameter | Description |
|---|---|
| `min_diversity` | Minimum number of unique IPs |
| `nameserver` | Filter by nameserver pattern |
| `domain` | Filter by domain name pattern |

In [ ]:
# ── 4.6 Search IP Diversity Patterns ─────────────────────────────────────────
params = {
    "min_diversity": 10,         # Minimum unique IPs seen
    # "nameserver": "ns1.evil*",  # Uncomment to filter by NS pattern
    # "domain": "*.xyz",          # Uncomment to filter by domain pattern
    "limit": 20
}

url = f"{BASE_URL}/api/v1/merge-api/explore/padns/search/ipdiversity"
data = sp_request("GET", url, params=params)
to_dataframe(data)  # Displays high-diversity domains as a table

### 4.7 ASNs Seen for Domain

Show all ASNs used by A records for a domain over the last 30 days, including subdomains.  
Useful for identifying hosting history and infrastructure changes.  
Docs: https://help.silentpush.com/docs/asns-seen-for-domain

```
GET https://app.silentpush.com/api/v1/merge-api/explore/padns/lookup/domain/asns/{domain}
```

In [ ]:
# ── 4.7 ASNs Seen for Domain ──────────────────────────────────────────────────
domain = "example.com"   # Domain to query (includes subdomains)

url = f"{BASE_URL}/api/v1/merge-api/explore/padns/lookup/domain/asns/{domain}"
sp_request("GET", url)

---
## Section 5 — Reputation & Risk History

Retrieve historical reputation scores for IPv4 addresses, nameservers, and subnets.  
These endpoints help track how the reputation of infrastructure has changed over time.

### 5.1 IPv4 Reputation History

Get the reputation history for a single IPv4 address, showing how scores have changed over time.  
Docs: https://help.silentpush.com/docs/ipv4-reputation-history

```
GET https://app.silentpush.com/api/v1/merge-api/explore/ipreputation/history/ipv4/{ipv4}
```

In [ ]:
# ── 5.1 IPv4 Reputation History ───────────────────────────────────────────────
ipv4 = "8.8.8.8"

url = f"{BASE_URL}/api/v1/merge-api/explore/ipreputation/history/ipv4/{ipv4}"
sp_request("GET", url)

### 5.2 Nameserver Reputation History

Get the reputation history for a nameserver — useful for identifying malicious DNS infrastructure.  
Docs: https://help.silentpush.com/docs/name-server-reputation-history

```
GET https://app.silentpush.com/api/v1/merge-api/explore/nsreputation/history/nameserver/{nameserver}
```

In [ ]:
# ── 5.2 Nameserver Reputation History ────────────────────────────────────────
nameserver = "ns1.example.com"   # Nameserver hostname

url = f"{BASE_URL}/api/v1/merge-api/explore/nsreputation/history/nameserver/{nameserver}"
sp_request("GET", url)

### 5.3 Subnet Reputation History

Get the reputation history for an IPv4 subnet (CIDR notation split into subnet + mask).  
Docs: https://help.silentpush.com/docs/subnet-reputation-history

```
GET https://app.silentpush.com/api/v1/merge-api/explore/ipreputation/history/subnet/{subnet}/{mask}
```

Example: For `192.168.1.0/24`, use `subnet=192.168.1.0` and `mask=24`.

In [ ]:
# ── 5.3 Subnet Reputation History ────────────────────────────────────────────
cidr = "8.8.8.0/24"             # Set your subnet in CIDR notation
subnet, mask = cidr.split("/")  # Auto-parsed — no manual splitting needed

url = f"{BASE_URL}/api/v1/merge-api/explore/ipreputation/history/subnet/{subnet}/{mask}"
sp_request("GET", url)

---
## Section 6 — SPQL Query Language

The **Silent Push Query Language (SPQL)** lets you search across Silent Push's scan data  
using a structured query syntax with support for complex filtering, field selection, and sorting.  
Docs: https://help.silentpush.com/docs/spql-api

> [!TIP]
> SPQL is the most flexible query interface Silent Push offers. Use `datasource=<name> AND ...` to scope queries to a specific data collection (`webscan`, `torscan`, `services`, `opendirectory`, `whois`). See **Section 6.2** for the full datasource reference.


### 6.1 SPQL Search Query

Submit an SPQL query to search across Silent Push scan data.

```
POST https://app.silentpush.com/api/v1/merge-api/explore/spql/search/
```

**Example SPQL syntax:**
```
domain = 'example.com'
asn = 15169 AND domain LIKE '%.google.%'
ip = '8.8.8.8'
```

| Body field | Description |
|---|---|
| `query` | SPQL query string |
| `limit` | Max results to return |
| `fields` | List of fields to include in results |

#### SPQL Query Reference

Copy any of these into the `SPQL_QUERY` variable below as a starting point.

| Use Case | SPQL Query |
|---|---|
| All scan data for a domain | `domain = 'example.com'` |
| All records for an IP | `ip = '8.8.8.8'` |
| Domains on a specific ASN | `asn = 15169 AND domain LIKE '%.google.%'` |
| Phishing keyword in domain | `domain LIKE '%paypal%' OR domain LIKE '%login%'` |
| Specific hosting provider | `hosting_provider LIKE '%DigitalOcean%'` |
| Domains on IP range | `ip LIKE '192.168.1.%'` |

Docs: https://help.silentpush.com/docs/spql-api

In [ ]:
# ── 6.1 SPQL Search Query ─────────────────────────────────────────────────────
# Change SPQL_QUERY to any query from the reference table above
SPQL_QUERY = "domain = 'example.com'"

query_body = {
    "query": SPQL_QUERY,
    "limit": 10            # Max results to return
    # "fields": ["domain", "ip", "asn"]  # Uncomment to select specific fields
}

url = f"{BASE_URL}/api/v1/merge-api/explore/spql/search/"
data = sp_request("POST", url, json=query_body)
to_dataframe(data)  # Displays scan results as a table

### 6.2 SPQL with Datasource Scoping

By default, SPQL searches across webscan data. Beware that each datasource has it's own specific queryable fields. Best practice is to create the SPQL query in the platform and then use the API to serve results as needed. You can scope a query to one or more specific datasources by prefixing with `datasource=<name> AND ...`.

| Datasource | What it contains |
|---|---|
| `webscan` | Standard web scan data — HTML, favicons, SSL certs, headers, redirect chains |
| `torscan` | Web scan data collected through the Tor network |
| `services` | Port/service scan data (open ports, banners, service fingerprints) |
| `opendirectory` | Detected open directory listings on web servers |
| `whois` | WHOIS registration data — registrant, registrar, creation/expiry dates |

**Syntax:**
```
# Single datasource
datasource=webscan AND favicon_murmur3 = 309020573

# Multiple datasources
datasource=webscan,torscan AND domain = 'example.com'

```

---
## Section 7 — Live Scanning

Perform real-time scans of URLs — retrieve rendered page content and screenshots  
from different geographic regions, platforms, and browsers.

### 7.1 Live Scan (v2)

Perform a live scan on a URL and retrieve the web scan result.  
Scans are executed from the specified region and simulated platform/browser.

```
GET https://app.silentpush.com/api/v2/live-scan/scan-on-demand/query/
```

| Parameter | Options |
|---|---|
| `url` | Full URL to scan |
| `platform` | `Desktop`, `Mobile`, `Crawler` |
| `OS` | `Windows`, `Linux`, `MacOS`, `iOS`, `Android` |
| `browser` | `Chrome`, `Firefox`, `Safari` |
| `region` | `US`, `EU`, `AS`, `TOR` |

In [ ]:
# ── 7.1 Live Scan (v2) ────────────────────────────────────────────────────────
params = {
    "url":      "https://example.com",  # URL to scan
    "platform": "Desktop",              # Desktop, Mobile, or Crawler
    "OS":       "Windows",              # Windows, Linux, MacOS, iOS, Android
    "browser":  "Chrome",               # Chrome, Firefox, Safari
    "region":   "US"                    # US, EU, AS, TOR
}

url = f"{BASE_URL}/api/v2/live-scan/scan-on-demand/query/"
sp_request("GET", url, params=params)

### 7.2 Live Screenshot (v2)

Generate a real-time screenshot of a URL. Returns a URL pointing to the captured screenshot image.

```
GET https://app.silentpush.com/api/v2/live-scan/screenshot-on-demand
```

In [ ]:
# ── 7.2 Live Screenshot (v2) ──────────────────────────────────────────────────
params = {
    "url": "https://example.com"   # URL to screenshot
}

url = f"{BASE_URL}/api/v2/live-scan/screenshot-on-demand"
data = sp_request("GET", url, params=params)

# Extract and render the screenshot inline in the notebook
if data:
    screenshot_url = (
        data.get("response", {}).get("screenshot_url")
        or data.get("screenshot_url")
    )
    if screenshot_url:
        print(f"Screenshot URL: {screenshot_url}")
        display(Image(url=screenshot_url))   # Renders the screenshot inline
    else:
        print("No screenshot URL found in response.")

---
## Section 8 — Data Exports

Download export files for IOFA feeds, organization data, bulk data, and IP context.  
Export file URLs are time-limited (valid for 3 hours after generation).  
The `{filename}` in each endpoint is obtained from the Silent Push portal or a prior API call.

### 8.1 IOFA Export

Download an IOFA (Indicator of Future Attack) export file.

```
GET https://app.silentpush.com/app/v1/export/iofa-exports/{filename}
```

In [ ]:
# ── 8.1 IOFA Export ───────────────────────────────────────────────────────────
filename = "your-iofa-export-filename.csv"   # Replace with actual filename from portal

url = f"{BASE_URL}/app/v1/export/iofa-exports/{filename}"
resp = requests.get(url, headers=HEADERS, timeout=60)
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type')}")

if resp.status_code == 200:
    save_export(resp.content, "iofa_export")  # Saves with auto-timestamped filename
else:
    print(f"Error: {resp.text[:500]}")

### 8.2 Organization Export

Download an organization-level export file.

```
GET https://app.silentpush.com/app/v1/export/organization-exports/{filename}
```

In [ ]:
# ── 8.2 Organization Export ───────────────────────────────────────────────────
filename = "your-org-export-filename.csv"   # Replace with actual filename from portal

url = f"{BASE_URL}/app/v1/export/organization-exports/{filename}"
resp = requests.get(url, headers=HEADERS, timeout=60)
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    save_export(resp.content, "org_export")
else:
    print(f"Error: {resp.text[:500]}")

### 8.3 Bulk Data Exports

Download a bulk data export file.

```
GET https://app.silentpush.com/app/v1/export/bulk-data-exports/{filename}
```

In [ ]:
# ── 8.3 Bulk Data Exports ─────────────────────────────────────────────────────
filename = "your-bulk-data-export-filename.csv"   # Replace with actual filename

url = f"{BASE_URL}/app/v1/export/bulk-data-exports/{filename}"
resp = requests.get(url, headers=HEADERS, timeout=60)
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    save_export(resp.content, "bulk_data_export")
else:
    print(f"Error: {resp.text[:500]}")

### 8.4 IP Context Export

Download an IP Context export file.

```
GET https://app.silentpush.com/api/v1/export/ip-context/{filename}
```

In [ ]:
# ── 8.4 IP Context Export ─────────────────────────────────────────────────────
filename = "your-ip-context-export-filename.csv"   # Replace with actual filename

url = f"{BASE_URL}/api/v1/export/ip-context/{filename}"
resp = requests.get(url, headers=HEADERS, timeout=60)
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    save_export(resp.content, "ip_context_export")
else:
    print(f"Error: {resp.text[:500]}")

---
## Section 9 — Customer Feed Management

Create and manage custom Silent Push threat feeds.  
Feeds can contain indicators (domains, IPs) that you curate and tag.  
Docs: https://help.silentpush.com/docs/view-the-customer-feed-api-endpoints

### 9.1 Create a Feed

Create a new customer feed. Returns the feed UUID which is required for subsequent feed operations.

```
POST https://app.silentpush.com/api/v1/feeds/
```

| Body field | Description |
|---|---|
| `name` | Display name for the feed |
| `description` | Description of what the feed contains |
| `source` | Source identifier for the feed |

In [ ]:
# ── 9.1 Create a Feed ─────────────────────────────────────────────────────────
feed_body = {
    "name":        "My Custom Threat Feed",
    "description": "Custom feed for tracking phishing infrastructure",
    "source":      "internal-research"
}

url = f"{BASE_URL}/api/v1/feeds/"
data = sp_request("POST", url, json=feed_body)

# Save the returned UUID for use in subsequent cells
FEED_UUID = data.get("uuid") or data.get("response", {}).get("uuid", "")
print(f"\nFeed UUID: {FEED_UUID}")

### 9.2 Add Indicators to a Feed

Add one or more indicators (domains, IPs) to an existing feed.

```
POST https://app.silentpush.com/api/v1/feeds/{feed_uuid}/indicators/
```

Request body: JSON array of indicator objects. Each object should include the indicator `name` and optional metadata.

In [ ]:
# ── 9.2 Add Indicators to a Feed ─────────────────────────────────────────────
# Set FEED_UUID from cell 9.1 or manually:
# FEED_UUID = "your-feed-uuid-here"

indicators = [
    {"name": "malicious-domain.xyz"},
    {"name": "192.0.2.100"},
    {"name": "phishing-site.net"}
]

url = f"{BASE_URL}/api/v1/feeds/{FEED_UUID}/indicators/"
sp_request("POST", url, json=indicators)

### 9.3 Add Tags to Indicators

Add tags to a specific indicator in a feed. Tags help categorize and filter indicators.

```
POST https://app.silentpush.com/api/v1/feeds/{feed_uuid}/indicators/{name}/tags/
```

| Parameter | Description |
|---|---|
| `feed_uuid` | UUID of the feed |
| `name` | Indicator name (domain or IP) |

Request body: JSON array of tag strings.

In [ ]:
# ── 9.3 Add Tags to Indicators ────────────────────────────────────────────────
# Set FEED_UUID from cell 9.1 or manually:
# FEED_UUID = "your-feed-uuid-here"

indicator_name = "malicious-domain.xyz"   # Indicator to tag
tags = ["phishing", "high-confidence", "2024-campaign"]

url = f"{BASE_URL}/api/v1/feeds/{FEED_UUID}/indicators/{indicator_name}/tags/"
sp_request("POST", url, json=tags)

---
## Section 10 — End-to-End Investigation Workflow

This section demonstrates how to chain multiple Silent Push APIs together to fully investigate a suspicious domain.  
**Set `TARGET` in the first cell below, then run all cells in order.**

**Workflow:**
1. Enrich the domain for risk scores and hosting context
2. Forward PADNS lookup to discover what IPs it resolves to
3. Enrich each resolved IP individually
4. Reverse PADNS on each IP to find co-hosted (potentially related) domains
5. Bulk risk score all co-hosted domains at once
6. SPQL deep-dive to surface scan data, cert info, and hosting context
7. Interpret the results

This mirrors how a real analyst would pivot from a single suspicious domain to a broader infrastructure picture.

> [!TIP]
> Change only the `TARGET` variable in cell 10.1, then use **Run All** on Section 10. Each step passes its results into the next — enrichment → DNS → co-hosted domains → bulk score → SPQL deep-dive.


In [ ]:
# ── 10.1 Set Investigation Target ─────────────────────────────────────────────
# Change this one variable and re-run all cells in Section 10
TARGET = "suspicious-example.com"

print(f"Investigation target: {TARGET}")
print("Run the cells below in order to build a complete picture of this domain's infrastructure.")

In [ ]:
# ── 10.2 Enrich the Target Domain ────────────────────────────────────────────
# Get risk score, ASN, hosting details, and reputation context
print(f"=== Step 1: Enrich domain {TARGET} ===")
domain_enrichment = sp_request("GET", f"{BASE_URL}/api/v1/merge-api/explore/enrich/domain/{TARGET}")

In [ ]:
# ── 10.3 Forward PADNS Lookup — Discover Resolved IPs ────────────────────────
# Find all IPv4 addresses this domain has resolved to in PADNS history
print(f"=== Step 2: Forward PADNS lookup for {TARGET} ===")
padns_fwd = sp_request(
    "GET",
    f"{BASE_URL}/api/v1/merge-api/explore/padns/lookup/query/a/{TARGET}",
    params={"limit": 50}
)

# Extract unique IPs from the response for use in subsequent steps
resolved_ips = []
if padns_fwd:
    records = padns_fwd.get("response", {})
    # Handle different response structures
    if isinstance(records, dict):
        for v in records.values():
            if isinstance(v, list):
                for r in v:
                    ip = r.get("answer") or r.get("ip") or r.get("value")
                    if ip and ip not in resolved_ips:
                        resolved_ips.append(ip)
    elif isinstance(records, list):
        for r in records:
            ip = r.get("answer") or r.get("ip") or r.get("value")
            if ip and ip not in resolved_ips:
                resolved_ips.append(ip)

print(f"\nResolved IPs found: {resolved_ips or '(none — check response above)'}")
to_dataframe(padns_fwd)

In [ ]:
# ── 10.4 Enrich Each Resolved IP ─────────────────────────────────────────────
# Get ASN, hosting provider, reputation, and geolocation for each IP
print(f"=== Step 3: Enrich {len(resolved_ips)} resolved IP(s) ===\n")
ip_enrichments = {}
for ip in resolved_ips[:10]:   # Cap at 10 IPs to avoid excessive API calls
    print(f"--- Enriching {ip} ---")
    result = sp_request("GET", f"{BASE_URL}/api/v1/merge-api/explore/enrich/ipv4/{ip}")
    ip_enrichments[ip] = result
    print()

In [ ]:
# ── 10.5 Reverse PADNS — Find Co-hosted Domains ──────────────────────────────
# For each IP, find all other domains that have resolved to it (potential infrastructure overlap)
print(f"=== Step 4: Reverse PADNS — co-hosted domains on resolved IPs ===\n")
cohosted_domains = set()
for ip in resolved_ips[:5]:   # Cap at 5 IPs for the investigation
    print(f"--- Domains co-hosted on {ip} ---")
    result = sp_request(
        "GET",
        f"{BASE_URL}/api/v1/merge-api/explore/padns/lookup/answer/a/{ip}",
        params={"limit": 50}
    )
    if result:
        records = result.get("response", {})
        candidates = records if isinstance(records, list) else next(
            (v for v in records.values() if isinstance(v, list)), []
        )
        for r in candidates:
            d = r.get("query") or r.get("domain") or r.get("name")
            if d and d != TARGET:
                cohosted_domains.add(d)
    to_dataframe(result)
    print()

print(f"Total unique co-hosted domains found: {len(cohosted_domains)}")
cohosted_list = sorted(cohosted_domains)
print(cohosted_list[:20])

In [ ]:
# ── 10.6 Bulk Risk Score — Co-hosted Domains ─────────────────────────────────
# Score all co-hosted domains at once to identify which ones are high-risk
print(f"=== Step 5: Bulk risk score for co-hosted domains ===")
# Include the original target plus up to 99 co-hosted domains (100 max per request)
domains_to_score = [TARGET] + cohosted_list[:99]
print(f"Scoring {len(domains_to_score)} domain(s)...\n")

risk_data = sp_request(
    "POST",
    f"{BASE_URL}/api/v1/merge-api/explore/bulk/domain/riskscore",
    json=domains_to_score
)

risk_df = to_dataframe(risk_data)  # Displays as a sortable table — sort by score column

# Highlight top risky domains if DataFrame was created
if risk_df is not None and not risk_df.empty:
    score_col = next((c for c in risk_df.columns if "score" in c.lower()), None)
    if score_col:
        print(f"\nTop 10 highest-risk domains (by {score_col}):")
        display(risk_df.sort_values(score_col, ascending=False).head(10))

In [ ]:
# ── 10.7 SPQL Deep-Dive ───────────────────────────────────────────────────────
# Run SPQL queries to surface scan data, certificate details, and hosting context
# that aren't returned by individual enrichment endpoints.
print(f"=== Step 6: SPQL deep-dive for {TARGET} ===\n")

# Query 1: All scan data for the target domain
print("--- SPQL: Scan data for target domain ---")
spql_domain = sp_request("POST", f"{BASE_URL}/api/v1/merge-api/explore/spql/search/", json={
    "query": f"domain = '{TARGET}'",
    "limit": 20
})
to_dataframe(spql_domain)

# Query 2: Scan data for all resolved IPs (surfaces hosting context + cert info)
if resolved_ips:
    ip_list = ", ".join(f"'{ip}'" for ip in resolved_ips[:5])
    print(f"\n--- SPQL: Scan data for resolved IPs ({', '.join(resolved_ips[:5])}) ---")
    spql_ips = sp_request("POST", f"{BASE_URL}/api/v1/merge-api/explore/spql/search/", json={
        "query": f"ip IN ({ip_list})",
        "limit": 20
    })
    to_dataframe(spql_ips)

### How to Interpret Investigation Results

After running all cells above, you should have:

| Step | Data Gathered | What to Look For |
|---|---|---|
| Domain Enrichment | Risk score, ASN, hosting provider | High risk score (>70), bulletproof hosting, newly registered |
| Forward PADNS | Historical IPs | Large number of IPs = fast-flux; frequent changes = evasion |
| IP Enrichment | ASN, reputation, geolocation | IPs in high-abuse ASNs, multiple countries, residential proxies |
| Reverse PADNS | Co-hosted domains | Many unrelated domains on same IP = shared malicious infrastructure |
| Bulk Risk Score | Scores for all co-hosted domains | Cluster of high-score domains confirms shared threat actor infra |
| SPQL Deep-Dive | Scan data, certs, headers | Self-signed certs, phishing page content, fake login forms |

**Red flags that suggest a confirmed threat:**
- Risk score > 70 on the target domain
- Multiple co-hosted domains also scoring > 70
- Fast-flux behavior (many IPs, short TTLs)
- Self-signed certificates on an otherwise consumer-facing site
- Hosting on known bulletproof providers

**Next steps if confirmed malicious:**
- Add to a Customer Feed (Section 9) for tracking
- Export indicators via IOFA Export (Section 8) for SIEM ingestion
- Run ThreatCheck (Section 1) on all co-hosted IPs to cross-reference IOFA feeds